# Notebook 1 - Intro Regression (Guided)

**Expected duration:** 60-90 minutes

## Objective
Build a first end-to-end regression workflow to predict hourly bike demand.

## Inputs
- UCI Bike Sharing data (`data/hour.csv`, auto-downloaded if missing)
- Calendar and weather features

## Outputs
- Baseline vs linear vs random-forest performance comparison
- Error interpretation and residual diagnostics

## Checkpoint expectations
- Detect leakage columns correctly
- Build preprocessing pipeline and compare models
- Explain MAE/RMSE in business terms


## Dataset handling

This notebook uses the **UCI Bike Sharing Dataset**, hourly version.

The notebook:
- first checks whether `data/hour.csv` already exists,
- if not, it downloads and extracts the dataset,
- then it loads the CSV into a DataFrame.

For a workshop with unreliable internet, place `hour.csv` in:

```text
data/hour.csv
```

before the session begins.

## Environment setup (run once outside class flow)
Install dependencies before class:

```bash
pip install numpy pandas matplotlib scikit-learn ipython
```


In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn ipython


## 0. Setup

In [ ]:
from pathlib import Path
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
def load_csv_from_uci_zip(url, suffix, local_path, sep=","):
    """Load a CSV from a local file if present; otherwise download and cache it."""
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        print(f"Loading cached dataset from: {local_path}")
        return pd.read_csv(local_path, sep=sep)

    print("Local file not found. Downloading dataset...")
    with urllib.request.urlopen(url) as response:
        raw_zip = response.read()

    with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
        matching_files = [name for name in zf.namelist() if name.endswith(suffix)]
        if not matching_files:
            raise FileNotFoundError(f"Could not find {suffix} inside the downloaded ZIP file.")

        selected_file = matching_files[0]
        with zf.open(selected_file) as src:
            content = src.read()

    local_path.write_bytes(content)
    print(f"Saved dataset to: {local_path}")
    return pd.read_csv(local_path, sep=sep)


def make_one_hot_encoder(dense=False):
    """Create a OneHotEncoder compatible with multiple scikit-learn versions."""
    if dense:
        try:
            return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=False)

    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def evaluate_regression(name, y_true, y_pred):
    """Return a one-row DataFrame with common regression metrics."""
    return pd.DataFrame([{
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R²": r2_score(y_true, y_pred)
    }])


def plot_actual_vs_predicted(y_true, y_pred, title):
    plt.figure(figsize=(7, 5))
    plt.scatter(y_true, y_pred, alpha=0.35)
    lower = min(np.min(y_true), np.min(y_pred))
    upper = max(np.max(y_true), np.max(y_pred))
    plt.plot([lower, upper], [lower, upper], linestyle="--")
    plt.xlabel("Actual rentals")
    plt.ylabel("Predicted rentals")
    plt.title(title)
    plt.show()


def plot_residuals(y_true, y_pred, title):
    residuals = y_true - y_pred
    plt.figure(figsize=(7, 5))
    plt.scatter(y_pred, residuals, alpha=0.35)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted rentals")
    plt.ylabel("Residual = actual - predicted")
    plt.title(title)
    plt.show()


## 1. Load the dataset

The target variable is hourly rental demand.

The data contains:
- calendar information,
- weather conditions,
- normalized continuous weather variables,
- rental counts.

In [ ]:
BIKE_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

bike = load_csv_from_uci_zip(
    BIKE_URL,
    suffix="hour.csv",
    local_path="data/hour.csv",
    sep=","
)

bike.head()

## 2. Basic dataset inspection

In [ ]:
print("Shape:", bike.shape)
print("\nColumns:")
print(list(bike.columns))
display(bike.head())
display(bike.describe(include="all").T)

### Quick data dictionary for coded variables
Use this mini-reference during EDA and interpretation.


In [ ]:
codebook = pd.DataFrame([
    ["season", "1=spring, 2=summer, 3=fall, 4=winter"],
    ["yr", "0=2011, 1=2012"],
    ["mnth", "1-12 month index"],
    ["weekday", "0=Sunday ... 6=Saturday"],
    ["workingday", "1=working day, 0=weekend/holiday"],
    ["weathersit", "1=clear, 2=mist/cloudy, 3=light rain/snow, 4=heavy rain/snow"],
], columns=["column", "meaning"])

display(codebook)


### Teaching note
Before modeling, learners should know:
- how many rows and columns exist,
- which column is the target,
- which features are numeric or categorical,
- whether any columns are suspiciously close to the target.

## 3. Check missing values and duplicates


In [ ]:
missing_summary = bike.isna().sum().sort_values(ascending=False)
duplicate_count = bike.duplicated().sum()

print("Missing values per column:")
display(missing_summary)

print(f"Duplicate rows: {duplicate_count}")

The dataset is relatively clean, which is useful pedagogically.  
We will still use imputation inside our preprocessing pipeline because:
1. it is good practice,
2. real datasets are often less tidy,
3. pipelines should be robust to missing values.

## 4. Frame the machine learning problem


In [ ]:
target = "cnt"

print("Target variable:", target)
print("Target description: total hourly bicycle rentals")
display(bike[[target]].describe())

### Why this is a regression problem
The target `cnt` is a **continuous numeric quantity**:
- 0 rentals,
- 25 rentals,
- 180 rentals,
- 500+ rentals.

Therefore, this is a **supervised regression** task.

## 5. Spot target leakage


In [ ]:
bike[["casual", "registered", "cnt"]].head()

The column `cnt` is defined as:

\[
cnt = casual + registered
\]

If we leave `casual` and `registered` in the feature matrix while trying to predict `cnt`, the model would receive the answer indirectly.

That is **data leakage**.

We remove:
- `cnt` because it is the target,
- `casual` and `registered` because they leak the target,
- `instant` because it is only a row identifier.

In [ ]:
drop_columns = ["cnt", "casual", "registered", "instant"]

X = bike.drop(columns=drop_columns)
y = bike[target]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Columns retained:")
print(list(X.columns))

### Learner checkpoint
Pause and ask learners:
- What would happen to model scores if we kept `casual` and `registered`?
- Why would that be misleading in a real deployment?


## 6. Explore the target distribution


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(y, bins=40)
plt.xlabel("Hourly rentals")
plt.ylabel("Number of observations")
plt.title("Distribution of hourly bike rentals")
plt.show()

print("Target summary:")
display(y.describe())

### Teaching note
The target is right-skewed:
- many hours have low to moderate rentals,
- fewer hours have very high demand.

This matters because:
- large errors on peak-demand hours may dominate RMSE,
- MAE and RMSE can tell slightly different stories.

## 7. Simple exploratory analysis


In [ ]:
eda = bike.copy()
eda["dteday"] = pd.to_datetime(eda["dteday"])

hourly_demand = eda.groupby("hr")["cnt"].mean()
weather_demand = eda.groupby("weathersit")["cnt"].mean()
weekday_demand = eda.groupby("weekday")["cnt"].mean()

display(hourly_demand.rename("average_rentals_by_hour"))
display(weather_demand.rename("average_rentals_by_weather_code"))
display(weekday_demand.rename("average_rentals_by_weekday"))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hourly_demand.index, hourly_demand.values, marker="o")
plt.xlabel("Hour of day")
plt.ylabel("Average rentals")
plt.title("Average bike rentals by hour")
plt.xticks(range(0, 24, 2))
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(weather_demand.index.astype(str), weather_demand.values)
plt.xlabel("Weather situation code")
plt.ylabel("Average rentals")
plt.title("Average bike rentals by weather situation")
plt.show()

### Teaching note
This makes the task more tangible:
- demand tends to vary strongly by **hour of day**,
- poor weather reduces demand,
- demand is not simply linear in one variable.

## 8. Feature engineering


In [ ]:
X_model = X.copy()

X_model["dteday"] = pd.to_datetime(X_model["dteday"])
X_model["day_of_year"] = X_model["dteday"].dt.dayofyear
X_model = X_model.drop(columns="dteday")

display(X_model.head())
print("Final feature columns:")
print(list(X_model.columns))


### Why engineer date features?
Models cannot directly learn from a raw date string.
We convert date into a useful numeric calendar signal:
- day of year.


## 9. Define feature groups


In [ ]:
categorical_features = [
    "season", "yr", "mnth", "hr",
    "holiday", "weekday", "workingday", "weathersit"
]

numeric_features = [
    col for col in X_model.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

### Important modeling detail
Several columns are stored as integers but should be treated as **categories**:
- hour,
- weekday,
- month,
- weather code,
- working-day flag.

Treating them as numeric in a linear model would imply artificial ordering and distances.

## 10. Build preprocessing pipelines


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_one_hot_encoder(dense=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor


### Why this preprocessing setup?
- Numeric columns are imputed then scaled.
- Categorical columns are imputed then one-hot encoded.
- The pipeline keeps all preprocessing reproducible and tied to model training.


## 11. Split into training and test sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

### Teaching note
The test set is kept untouched until final evaluation.  
Model selection and hyperparameter tuning happen using only the training data.

## 12. Baseline model


In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_results = evaluate_regression(
    "Mean baseline",
    y_test,
    baseline_pred
)

baseline_results


### Why the baseline matters
A baseline answers:

> Is our ML model better than a trivial default?

Here, the baseline always predicts the **average training-set demand**.

## 13. Linear regression


In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_results = evaluate_regression(
    "Linear regression",
    y_test,
    linear_pred
)

linear_results


### Why linear regression?
It is:
- simple,
- interpretable,
- a useful first real model.

But bike rental demand is likely non-linear:
- rush-hour demand,
- weather interactions,
- seasonality.

## 14. Random forest regression


In [ ]:
forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=180,
        max_depth=None,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)

forest_results = evaluate_regression(
    "Random forest",
    y_test,
    forest_pred
)

forest_results


### Why a random forest?
Random forests can capture:
- non-linear patterns,
- feature interactions,
- complex split-based decision rules.

They are often much stronger than a linear model on structured/tabular data.

## 15. Compare core models


In [ ]:
comparison = pd.concat(
    [
        baseline_results,
        linear_results,
        forest_results,
    ],
    ignore_index=True
).sort_values("RMSE")

comparison


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison["Model"], comparison["RMSE"])
plt.ylabel("RMSE")
plt.title("Model comparison by RMSE")
plt.xticks(rotation=20)
plt.show()

### Interpreting the metrics
- **MAE**: average absolute error, easy to explain.
- **RMSE**: penalizes larger errors more strongly.
- **R²**: proportion of variance explained compared with a mean-only model.

For operational forecasting, RMSE is useful when large misses are especially costly.

### Business interpretation of error
If the best model has MAE around 25 to 35, that means the prediction is off by about **25 to 35 bikes per hour on average**.


In [ ]:
best_row = comparison.iloc[0]
print(
    f"Best intro model: {best_row['Model']} | "
    f"MAE={best_row['MAE']:.2f}, RMSE={best_row['RMSE']:.2f}, R?={best_row['R?']:.3f}"
)


### Learner checkpoint
Ask learners:
- Which metric would you prioritize for city bike planning: MAE, RMSE, or R??
- Why might RMSE be useful when big misses are costly?


## 16. Diagnose predictions visually


In [ ]:
plot_actual_vs_predicted(
    y_test,
    forest_pred,
    "Random Forest: Actual vs Predicted"
)


In [ ]:
plot_residuals(
    y_test,
    forest_pred,
    "Random Forest: Residual Plot"
)


### What to look for
- If predictions cluster near the diagonal, the model is doing well.
- Systematic curves or fan shapes in residuals suggest remaining structure or heteroscedasticity.
- Large residuals during peak-demand periods may deserve closer inspection.

## 17. Summary


By the end of this intro notebook, we have:
- framed bike demand prediction as a regression task,
- removed leakage columns,
- engineered practical calendar features,
- built reusable preprocessing pipelines,
- compared baseline, linear, and random-forest models,
- interpreted model error in practical terms,
- diagnosed predictions with residual plots.


## 18. Suggested live-teaching pauses


Use these pause points during delivery:

1. **After leakage detection**  
   Ask: ?What would happen if we kept `casual` and `registered`??

2. **After linear regression results**  
   Ask: ?Why might a linear model struggle here??

3. **After model comparison**  
   Ask: ?Do we care more about average misses or big misses??

4. **After residual plot**  
   Ask: ?Where does the model still struggle??
